**Building Models in PyTorch**

`torch.nn.Module` and `toch.nn.Parameter`

In this video, we'll be discussing some of the tools PyTorch makes available for building deep learning networks.

Except for `Parameter` all of them are subclasses of `toch.nn.Module`. This PyTorch base class encapsulate behaviours specific to PyTorch Models and components.

`toch.nn.Module` is registering parameters. If a `Module` sublass has learning weights, these weigts expressed as isntances of `torch.nn.Parameter`. `Parameter` subclass of `torch.Tensor`, with special behaviour they are assigned as attributes of a `Module`, they are added to list of module parametsr. May be accessed thorugh the `parameters()` method on the `Module` class

Simple example, very simple model with two linear layers and an activation function. Create an instance of it and ask it to report on its parameters.



In [2]:
import torch

class TinyModel(torch.nn.Module):

    def __init__(self):
        super(TinyModel, self).__init__()

        self.linear1=torch.nn.Linear(100,200)
        self.activation1=torch.nn.ReLU()
        self.linear2=torch.nn.Linear(200,10)
        self.softmax=torch.nn.Softmax()
    def forward(self,x):
        x=self.linear1(x)
        x=self.activation1(x)
        x=self.softmax(x)
        return x

tinymodel = TinyModel()

print('The model:')
print(tinymodel)

print('\n\nJust on layer:')
print(tinymodel.linear2)

print('\n\nThe parameters of the model:')
for param in tinymodel.parameters():
    print(param)

print('\n\nLayer params:')
for param in tinymodel.linear2.parameters():
    print(param)

The model:
TinyModel(
  (linear1): Linear(in_features=100, out_features=200, bias=True)
  (activation1): ReLU()
  (linear2): Linear(in_features=200, out_features=10, bias=True)
  (softmax): Softmax(dim=None)
)


Just on layer:
Linear(in_features=200, out_features=10, bias=True)


The parameters of the model:
Parameter containing:
tensor([[ 0.0690,  0.0055,  0.0329,  ..., -0.0746,  0.0048,  0.0829],
        [ 0.0187, -0.0414,  0.0846,  ..., -0.0695, -0.0110,  0.0063],
        [ 0.0235,  0.0497,  0.0379,  ...,  0.0851,  0.0885,  0.0386],
        ...,
        [ 0.0269,  0.0496, -0.0476,  ...,  0.0774, -0.0824,  0.0056],
        [ 0.0345, -0.0652, -0.0288,  ...,  0.0716,  0.0473, -0.0637],
        [-0.0066,  0.0510, -0.0172,  ..., -0.0691, -0.0300, -0.0986]],
       requires_grad=True)
Parameter containing:
tensor([-0.0428, -0.0253, -0.0112,  0.0031, -0.0725,  0.0893, -0.0550,  0.0729,
        -0.0828,  0.0079,  0.0869, -0.0511,  0.0785, -0.0985, -0.0599,  0.0532,
        -0.0098, -0.0286,

**Common Layer Types**
**Linear Layers**

The most basic type of neural networth is *linear* or *fully connected* layer. This is a layer where every input influence every output to layer's weights. M inputs, N outputs, weights will be M*N matrix

In [3]:
line = torch.nn.Linear(3,2)
x=torch.rand(1,3)
print('Input:')
print(x)

print('\n\nWeight and Bias parameters:')
for param in line.parameters():
    print(param)

y = line(x)
print('\n\nOutput:')
print(y)

Input:
tensor([[0.8447, 0.4476, 0.4778]])


Weight and Bias parameters:
Parameter containing:
tensor([[-0.4676, -0.2024, -0.4907],
        [ 0.3087, -0.3086,  0.1913]], requires_grad=True)
Parameter containing:
tensor([-0.4556, -0.3449], requires_grad=True)


Output:
tensor([[-1.1756, -0.1309]], grad_fn=<AddmmBackward0>)


matrix multiplication of `x` by linear weights, and add biases, et output vector `y`

one other important feature to note: When we checked the weights of our layer with `lin.weight`, it reported itself as a `Parameter`, which is subclass of `tensor`. Lets us know it's tracking autograd. Default behavior for `Parameter` different from `Tensor`.

Used widely in deep learning models. Classifier models, have n outputs. 

**Convolutional Layers**

Built to handle data with high degree of spatial correlation. Commonly used in computer vision, detect close gropuings of features which compose hiehger-level features. Pop up in NLPs where word immediate context can affect meaning of sentences. 

Saw convulational layers in action in LeNet5

In [4]:
import torch.functional as F

class LeNet(torch.nn.Module):

    def __init__(self):
        super(LeNet, self).__init__()

        self.conv1=torch.nn.Conv2d(1,6,5)
        self.conv2=torch.nn.Conv2d(6,16,3)
        
        self.fc1=torch.nn.Linear(16*6*6,120)
        self.fc2=torch.nn.Linear(120,84)
        self.fc3=torch.nn.Linear(84,10)

    def forward(self,x):
        x=F.max_pool2d(F.relu(self.conv1(x)),(2,2))
        x=F.max_pool2d(F.relu(self.conv2(x)),2)
        x=x.view(-1,self.num_flat_features(x))
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=self.fc3(x)
        return x
    
    def num_flat_features(self,x):
        size = x.size()[1:] # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

**Reccurent Layers**

RNNs used for sequential data - anything from time-series measurements from a scientific instrument to natural language sentences to DNA nucleotides. RNN maintain hidden state that is memory for what is has seen in sequence so far. Internal structure of RNN layer the LSTM (long short-term memory) abd GRU (gated recurrent unit) moderately complex and beyond scope. Example is LSTM part of speech tagger (verb, noun, etc. )

In [ ]:
class LSTMTagger(torch.nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = torch.nn.Embedding(vocab_size, embedding_dim)
        #LSTM takes word embeddings as inputs, outputs hidden states
        # with dimensionality hidden_dim
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim)
        # Linear layer maps from hidden state space to tag space
        self.hidden2tag = torch.nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = torch.nn.functional.log_softmax(tag_space, dim=1)
        return tag_scores

constructor four arguments:
- `vocab_size` # of words in input vocab. Each word is a one hot vector, in a `vocab_size-dimensional space.
- `tagset_size` number of tags in output set
- `embedding_dim` size of embedding space for vocab. embedding maps a vocab onto low-dimensional space
- `hidden_dim` size of LSTMs memory

input will be sentence with words represented as indices of on-hot vectors. Embedding layer map these to `embedding_dim` dimensional space. LSTM takes sequence of embedding iterate over it, fielding output vector of length `hidden_dim`. final layer is classifier, applying `log_softmax()` to output of final layer converts output into a normalized set of estimated probability. 

**Transformers**

multi-purpose networks taken over state of the art in NLP with models like BERT. PyTorch has `Transformer` class that allows you to define overall parameters of a transformer model. # of attention heads, encoder decoder layers, dropout activation function. `torch.nn.Transformer` class also has classes to encapsulate individual components (`TransformerEncoder`, `TransformerDecoder`) and subcomponents (`TransformerEncoderLayer`, `TransformerDecoderLayer`). 

**Other Layers and Functions**

**Data Manipulatin Layers**

Other important functions that don't participate in learning process.

**Max Pooling** and twin min pooling reduce tensory by combining cells, assigning max value of input cells to output cells


In [6]:
my_tensor=torch.rand(1,6,6)
print(my_tensor)

maxpool_layer = torch.nn.MaxPool2d(3)
print(maxpool_layer(my_tensor))

tensor([[[0.5699, 0.9091, 0.7635, 0.6464, 0.2203, 0.9003],
         [0.2390, 0.4728, 0.3130, 0.5901, 0.1806, 0.7663],
         [0.6243, 0.8930, 0.9192, 0.5880, 0.0502, 0.2802],
         [0.4963, 0.6679, 0.7907, 0.3049, 0.5276, 0.9083],
         [0.7698, 0.6496, 0.9588, 0.9898, 0.8626, 0.3788],
         [0.5904, 0.2112, 0.7743, 0.1708, 0.2439, 0.7630]]])
tensor([[[0.9192, 0.9003],
         [0.9588, 0.9898]]])


look at values above, each value of maxpooled output is the max value of each quadrant of 6x6 output.

**Normalization layers** re-center and normalize output of one layer before feeding it ot another. Centering and scaling intermediate tensors has a number of beneficial effects, such as letting you use higher learning rates without exploding/vanishing gradients.



In [7]:
my_tensor=torch.rand(1,4,4)*20+5
print(my_tensor)

print(my_tensor.mean())

norm_layer=torch.nn.BatchNorm1d(4)
normed_layer = norm_layer(my_tensor)
print(normed_layer)

print(normed_layer.mean())

tensor([[[19.4955, 20.0772, 19.2844, 17.1708],
         [ 6.2731, 22.4109,  9.2522, 17.7760],
         [13.6599, 19.9021, 23.6968, 17.0366],
         [22.9984, 11.5307, 13.1118,  8.5843]]])
tensor(16.3913)
tensor([[[ 0.4445,  0.9736,  0.2524, -1.6705],
         [-1.1839,  1.3120, -0.7232,  0.5951],
         [-1.3310,  0.3598,  1.3876, -0.4164],
         [ 1.6522, -0.4666, -0.1745, -1.0110]]],
       grad_fn=<NativeBatchNormBackward0>)
tensor(-9.6858e-08, grad_fn=<MeanBackward0>)


Running the cell above, added large scaling factor and offset to an input tensor; you should see the input tensor's `mean()` somewhere in the nieghbouhood of 15. Group around 0. Mean should be very small. 

Beneficial because many activation functions have strongest gradients near 0. Sometimes suffer from vanishing or exploding gradients. Keeping data centered around area of steepest gradient means it tends to learn faster. 

**Dropout layers** are a tool for encouraging sparse representations in your model. that is pushing it to do inference with less data. 

Work by randomly setting parts of the input tensor *during training* - turned off for inference. Forces model to learn against this masked or reduced dataset. 

In [8]:
my_tensor = torch.rand(1,4,4)
print(my_tensor)

dropout=torch.nn.Dropout(p=0.4)
print(dropout(my_tensor))
print(dropout(my_tensor))

tensor([[[0.4005, 0.2156, 0.3865, 0.2134],
         [0.3516, 0.8828, 0.3425, 0.3676],
         [0.4437, 0.1370, 0.9878, 0.9046],
         [0.6560, 0.8700, 0.3531, 0.4316]]])
tensor([[[0.6675, 0.3594, 0.0000, 0.3556],
         [0.0000, 1.4714, 0.5708, 0.6127],
         [0.7395, 0.0000, 1.6463, 1.5076],
         [1.0934, 1.4500, 0.5885, 0.0000]]])
tensor([[[0.6675, 0.0000, 0.6441, 0.3556],
         [0.5860, 1.4714, 0.0000, 0.6127],
         [0.0000, 0.2284, 0.0000, 1.5076],
         [0.0000, 0.0000, 0.5885, 0.7193]]])


**Activation Functions**

Make deep learning possible. Program with many parameters, *simulates a mathematical function*. If all we did was multiple tensors by layer weights repeatedly, only simulate *linear* functions. further, no point to having many layers. Inserting non-linear activation functions between layers is what allows deep learning.

`toch.nn.Module` has objects encapsulating all of the major activation functions including ReLU and its many variants, Tanh, Hardtanh, sigmoid, and more. Also includes other functions such as Softmax, that are most useful at output stage of model 

**Loss Functions**

loss functions tell us how far model prediction from correct answer. Container variety of loss functions, including common MSE (mean squared error=L2 norm), cross entropy loss, and negative likelihood loss (useful for classifiers) and others. 